This is where I experiment with features.

In [ ]:
%load_ext line_profiler

In [1]:
from src.script import learning, load_object, MultilayerPerceptron, cost, accuracy, cost_gradient, vectorize_learning_set
from src.dataset_reader import load_dataset
import numpy as np
import matplotlib.pyplot as plt
import random
from copy import deepcopy
from timeit import timeit
multilayer_perceptron = MultilayerPerceptron(layout=[784, 16, 16, 10], dtype=np.float64)
training_set, test_set = load_dataset(training_size=60000, old_format=True)
training_costs, accuracies, gradients_norms, consecutive_gradients_cosines = [], [], [], []

c:\Users\anato\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
tracked_metrics = learning(multilayer_perceptron=multilayer_perceptron, training_set=training_set, test_set=test_set, eta=1.5*1e-03, steps_number=300, metrics_to_track=["training_costs", "accuracies", "gradients_norms", "consecutive_gradients_cosines"])
training_costs += tracked_metrics["training_costs"]
accuracies += tracked_metrics["accuracies"]
gradients_norms += tracked_metrics["gradients_norms"]
consecutive_gradients_cosines += tracked_metrics["consecutive_gradients_cosines"]

In [ ]:
training_costs

In [ ]:
plt.close()
fig, ax1 = plt.subplots()

ax1.plot(training_costs[300:], label="training_costs", color="blue")
ax1.legend(loc = "upper left")
ax1.set_ylim([0,1])
"""
ax2 = ax1.twinx()
ax2.plot(consecutive_gradients_cosines[0:], label="previous_cosines", color="red")
ax2.legend(loc="upper right")
ax2.set_ylim([-1,1])

ax3 = ax2.twinx()
ax3.plot(gradients_norms[0:], label="previous_gradients_norms", color="purple")
ax3.legend(loc = "lower left")"""

plt.show()

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(accuracies[300:], label="previous_accuracies", color="green")
ax1.legend(loc = "upper left")
ax1.set_ylim([0,1])
"""
ax2 = ax1.twinx()
ax2.plot(previous_cosines[0:], label="previous_cosines", color="red")
ax2.legend(loc="upper right")
ax2.set_ylim([-1,1])

ax3 = ax1.twinx()
ax3.plot(previous_gradient_norms[3:], label="previous_gradient_norms", color="purple")
ax3.legend(loc = "lower left")"""

plt.show()

## 1st experiment : Are training_cost and accuracy smooth at the scales uf the current prefered learning step (eta = 8*1e-03) ?##

In [ ]:
print(cost(multilayer_perceptron=multilayer_perceptron, training_set=training_set))
print(accuracy(multilayer_perceptron=multilayer_perceptron, test_set=test_set))

In [ ]:
in_between_costs = []
in_between_accuracies = []
eta = 1.3 * 1e-03
_training_set = random.choices(population=training_set, k=1000)
_cost_gradient = cost_gradient(
        multilayer_perceptron=multilayer_perceptron, training_set=_training_set
    )
N = len(multilayer_perceptron.layers)
layout = multilayer_perceptron.layout
nb_in_between_stops = 10
nudge = 1 / nb_in_between_stops

for _ in range(nb_in_between_stops+1):
    # modif. of the parameters
    for i in range(N):
        for variable in range(2):
            multilayer_perceptron.variables[i][variable] += (
                - nudge * eta * np.average(layout) * _cost_gradient[i][variable]
            )
            
    training_cost = cost(multilayer_perceptron=multilayer_perceptron, training_set=_training_set)
    in_between_costs.append(training_cost)
    _accuracy = accuracy(multilayer_perceptron=multilayer_perceptron, test_set=test_set)
    in_between_accuracies.append(_accuracy)

training_costs.append(in_between_costs[-1])
accuracies.append(in_between_accuracies[-1])
    

In [ ]:
fig1, ax1 = plt.subplots()

ax1.plot(in_between_costs, label="in_between_costs", color="blue")
ax1.legend(loc="upper right")
#ax1.set_ylim([0.4,0.6])

ax2 = ax1.twinx()
ax2.plot(in_between_accuracies, label="in_between_accuracies", color="green")
ax2.legend(loc="upper left")
#ax2.set_ylim([0.6,0.7])

plt.plot()

## 2nd experiment : Adding inertia ##
I'm going to compare the same mlp with and without inertia. First one by one, to get an intuition of it. Then, maybe later, I'll verify it by computing statistics on a lot of examples. 

In [ ]:
training_costs_with_inertia, accuracies_with_inertia = [], []
training_costs_with_inertia_2, accuracies_with_inertia_2 = [], []
multilayer_perceptron_with_inertia = deepcopy(multilayer_perceptron)
multilayer_perceptron_with_inertia_2 = deepcopy(multilayer_perceptron)

In [ ]:
_steps_number=400
eta = 2*1e-03
tracked_metrics = learning(multilayer_perceptron=multilayer_perceptron, training_set=training_set, test_set=test_set, eta=eta, steps_number=_steps_number, inertia=False, metrics_to_track=["training_costs", "accuracies"])
training_costs += tracked_metrics["training_costs"]
accuracies += tracked_metrics["accuracies"]
tracked_metrics_with_inertia = learning(multilayer_perceptron=multilayer_perceptron_with_inertia, training_set=training_set, test_set=test_set, eta=eta, steps_number=_steps_number, inertia_strength=0.3, metrics_to_track=["training_costs", "accuracies"])
training_costs_with_inertia += tracked_metrics_with_inertia["training_costs"]
accuracies_with_inertia += tracked_metrics_with_inertia["accuracies"]
tracked_metrics_with_inertia_2 = learning(multilayer_perceptron=multilayer_perceptron_with_inertia_2, training_set=training_set, test_set=test_set, eta=eta, steps_number=_steps_number, inertia_strength=0.6, metrics_to_track=["training_costs", "accuracies"])
training_costs_with_inertia_2 += tracked_metrics_with_inertia_2["training_costs"]
accuracies_with_inertia_2 += tracked_metrics_with_inertia_2["accuracies"]

In [ ]:
fig1, ax1 = plt.subplots()

ax1.plot(training_costs, label="training_costs", color="violet")
ax1.legend(loc="upper left")
ax1.set_ylim([0,1])

ax2 = ax1.twinx()
ax2.plot(training_costs_with_inertia, label="//_with_inertia (inertia_strength=0.3)", color="blue")
ax2.legend(loc="upper right")
ax2.set_ylim([0,1])

ax3 = ax1.twinx()
ax3.plot(training_costs_with_inertia_2, label="//_with_inertia (inertia_strength=0.6)", color="cyan")
ax3.legend(loc="lower left")
ax3.set_ylim([0,1])

plt.title("eta=2*1e-03 up until 400, exp. n°3")
plt.show()

In [ ]:
fig1, ax1 = plt.subplots()

ax1.plot(accuracies[:], label="accuracies", color="lime")
ax1.legend(loc="upper left")
ax1.set_ylim([0,1])

ax2 = ax1.twinx()
ax2.plot(accuracies_with_inertia[:], label="//_with_inertia (inertia_strength=0.3)", color="darkgreen")
ax2.legend(loc="upper right")
ax2.set_ylim([0,1])

ax3 = ax1.twinx()
ax3.plot(accuracies_with_inertia_2[:], label="//_with_inertia (inertia_strengthen=0.6)", color="yellow")
ax3.legend(loc="lower left")
ax3.set_ylim([0,1])

plt.title("eta=2*1e-03 up until 400, exp. n°3")
plt.show()

In [ ]:
print(np.average(accuracies[-110:-100]))
print(np.average(accuracies[-10:]))
print(np.average(accuracies_with_inertia[-110:-100]))
print(np.average(accuracies_with_inertia[-10:]))
print(np.average(accuracies_with_inertia_2[-110:-100]))
print(np.average(accuracies_with_inertia_2[-10:]))

## 3rd experiment : Vectorizing the learning sets in batches ##
Let's compare the speed of accuracy() with and without vectorization of the test set.


In [ ]:
vectorized_test_set = vectorize_learning_set(learning_set=test_set, max_batch_size = 10)

In [ ]:
# The globals parameter causes the code to be executed winthin the current workspace.
print(timeit(stmt="accuracy(multilayer_perceptron=multilayer_perceptron, test_set=test_set)", globals=globals(), number=10))
print(timeit(stmt="accuracy(multilayer_perceptron=multilayer_perceptron, test_set=vectorized_test_set)", globals=globals(), number=10))

In [ ]:
accuracy(multilayer_perceptron=multilayer_perceptron, test_set=test_set) == accuracy(multilayer_perceptron=multilayer_perceptron, test_set=vectorized_test_set)

In [ ]:
batch_sizes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
times_taken = []
for batch_size in batch_sizes:
    vectorized_test_set = vectorize_learning_set(learning_set=test_set, max_batch_size=batch_size)
    times_taken.append(timeit(stmt="accuracy(multilayer_perceptron=multilayer_perceptron, test_set=vectorized_test_set)", number=10, globals=globals()))

In [ ]:
plt.scatter(x=batch_sizes, y=times_taken)
plt.title(label="Accuracy() time vs batch size")
plt.show()

In [ ]:
batch_sizes = [1,4500, 5000, 5500, 6000]
times_taken = []
for batch_size in batch_sizes:
    vectorized_test_set = vectorize_learning_set(learning_set=test_set, max_batch_size=batch_size)
    times_taken.append(timeit(stmt="accuracy(multilayer_perceptron=multilayer_perceptron, test_set=vectorized_test_set)", number=100, globals=globals()))

In [ ]:
times_taken

Ccl : sur ma machine, il semblerait que la batch_size optimale pour calculer l'accuracy d'un mlp (784-16-16-10) (et avec tous les paramètres que j'ai fixés : le fait qu'il soit fully connected, that relu and the sigmoid are used) soit 5000. The computation is accelerated by a 175 factor between batch_size = 1 and batch_size = optimal_batch_size ! It was worth it.

In [3]:
multilayer_perceptron_copy = deepcopy(multilayer_perceptron)
vectorized_test_set = vectorize_learning_set(learning_set=test_set, max_batch_size=5000)
print(timeit(stmt="learning(multilayer_perceptron=multilayer_perceptron, training_set=training_set, eta=4*1e-03, steps_number=100, metrics_to_track=[\"accuracies\"], test_set = vectorized_test_set)", number=1, globals=globals()))
print(timeit(stmt="learning(multilayer_perceptron=multilayer_perceptron_copy, training_set=training_set, eta=4*1e-03, steps_number=100, metrics_to_track=[\"accuracies\"], test_set=test_set)", number=1, globals=globals()))

100%|██████████| 100/100 [01:29<00:00,  1.11it/s]


90.10096029995475


100%|██████████| 100/100 [04:01<00:00,  2.42s/it]

243.79584949999116


In [4]:
print(accuracy(multilayer_perceptron=multilayer_perceptron, test_set=vectorized_test_set))
print(accuracy(multilayer_perceptron=multilayer_perceptron_copy, test_set=vectorized_test_set))

0.6820999999999999
0.6893


Funny, not eaxctly the same. I don't see why. I don't think it comes from my code ?